# SafeView Model Training

Train an EfficientNet-B0 model to classify images into 4 categories:
- **Gore** — violent/bloody content
- **Horror** — dark, scary, disturbing imagery
- **NSFW** — nudity, pornographic content
- **Safe** — neutral, safe-for-work content

**Requirements:** Run on Google Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)

The trained model is exported to ONNX format and quantized for use in the SafeView browser extension (~5-7MB).

**Output tensor names:** `pixel_values` (input) / `logits` (output) — matching the extension's sandbox.js expectations.

In [ ]:
# Cell 1: Install dependencies & check GPU
!pip install -q torch torchvision onnx onnxruntime pillow scikit-learn matplotlib seaborn datasets

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Cell 2: Download datasets from HuggingFace Hub
# Using HuggingFace datasets library — reliable, no quota issues
import os
import shutil
import random
from pathlib import Path
from datasets import load_dataset
from PIL import Image

BASE_DIR = Path('dataset')
TRAIN_DIR = BASE_DIR / 'train'
VAL_DIR = BASE_DIR / 'val'
CLASSES = ['gore', 'horror', 'nsfw', 'safe']
VAL_RATIO = 0.2
MAX_PER_CLASS = 5000  # Cap per class for balanced training

# Create directory structure
for split_dir in [TRAIN_DIR, VAL_DIR]:
    for cls in CLASSES:
        (split_dir / cls).mkdir(parents=True, exist_ok=True)

def save_split(images, dest_class, prefix="img"):
    """Shuffle images and split into train/val."""
    random.seed(42)
    random.shuffle(images)
    images = images[:MAX_PER_CLASS]
    split_idx = int(len(images) * (1 - VAL_RATIO))
    
    for i, img in enumerate(images[:split_idx]):
        if isinstance(img, Image.Image):
            img.save(TRAIN_DIR / dest_class / f"{prefix}_{i}.jpg")
        else:
            shutil.copy2(img, TRAIN_DIR / dest_class / f"{prefix}_{i}.jpg")
    
    for i, img in enumerate(images[split_idx:]):
        if isinstance(img, Image.Image):
            img.save(VAL_DIR / dest_class / f"{prefix}_{i}.jpg")
        else:
            shutil.copy2(img, VAL_DIR / dest_class / f"{prefix}_{i}.jpg")
    
    print(f"  {dest_class}: {split_idx} train, {len(images) - split_idx} val")

# ============================================================
# 1. NSFW dataset — using Falconsai training data concept
# ============================================================
print("📥 Downloading NSFW dataset...")
try:
    nsfw_ds = load_dataset("zanderlewis/nsfw-selfharm-128", split="train")
    nsfw_images = [row['image'] for row in nsfw_ds if row.get('label') == 1]  # NSFW labeled
    safe_from_nsfw = [row['image'] for row in nsfw_ds if row.get('label') == 0]  # Safe labeled
    print(f"  Found {len(nsfw_images)} NSFW, {len(safe_from_nsfw)} safe images")
except Exception as e:
    print(f"  Primary NSFW dataset failed: {e}")
    print("  Trying alternative...")
    nsfw_ds = load_dataset("deepghs/nsfw_detect", split="train", streaming=True)
    nsfw_images = []
    safe_from_nsfw = []
    for i, row in enumerate(nsfw_ds):
        if i >= 10000:
            break
        if row.get('label', '') in ['nsfw', 'explicit', 'porn', 'hentai']:
            nsfw_images.append(row['image'])
        elif row.get('label', '') in ['safe', 'neutral', 'drawing']:
            safe_from_nsfw.append(row['image'])
    print(f"  Collected {len(nsfw_images)} NSFW, {len(safe_from_nsfw)} safe images")

save_split(nsfw_images, 'nsfw', prefix='nsfw')

# ============================================================
# 2. Gore / Violence dataset
# ============================================================
print("\n📥 Downloading Gore/Violence dataset...")
try:
    gore_ds = load_dataset("Tokymin/violence_detect_dataset", split="train")
    gore_images = [row['image'] for row in gore_ds if row.get('label') == 1]  # violent
    safe_from_gore = [row['image'] for row in gore_ds if row.get('label') == 0]  # non-violent
    print(f"  Found {len(gore_images)} violent, {len(safe_from_gore)} non-violent images")
except Exception as e:
    print(f"  Primary gore dataset failed: {e}")
    print("  Trying alternative...")
    gore_ds = load_dataset("miltoncandela/violence-detection-benchmark", split="train")
    gore_images = [row['image'] for row in gore_ds if 'violence' in str(row.get('label', '')).lower()]
    safe_from_gore = [row['image'] for row in gore_ds if 'violence' not in str(row.get('label', '')).lower()]
    print(f"  Collected {len(gore_images)} violent images")

save_split(gore_images, 'gore', prefix='gore')

# ============================================================
# 3. Horror dataset — dark/scary imagery
# ============================================================
print("\n📥 Preparing Horror dataset...")
# Horror is the hardest category to find clean datasets for.
# Strategy: use a subset of scary/dark images from available sources
# or manually curate from horror movie frames.
try:
    horror_ds = load_dataset("Quangnguyen711/Horror_Movie_Poster", split="train")
    horror_images = [row['image'] for row in horror_ds]
    print(f"  Found {len(horror_images)} horror images")
except Exception as e:
    print(f"  Horror poster dataset failed: {e}")
    horror_images = []

if len(horror_images) < 200:
    print("  ⚠️ Not enough horror images for robust training.")
    print("  Options:")
    print("  1. Upload horror images to 'dataset/train/horror/' and 'dataset/val/horror/' manually")
    print("  2. Continue with limited horror data (model may underperform on horror)")
    print(f"  Currently have: {len(horror_images)} images")

if horror_images:
    save_split(horror_images, 'horror', prefix='horror')

# ============================================================
# 4. Safe dataset — combine safe images from multiple sources
# ============================================================
print("\n📥 Preparing Safe dataset...")
safe_images = safe_from_nsfw + safe_from_gore

# Add more safe images if needed
if len(safe_images) < 2000:
    print("  Adding general safe images from CIFAR concepts...")
    try:
        places_ds = load_dataset("tanganke/places365", split="train", streaming=True)
        extra_safe = []
        for i, row in enumerate(places_ds):
            if i >= 3000:
                break
            extra_safe.append(row['image'])
        safe_images.extend(extra_safe)
        print(f"  Added {len(extra_safe)} extra safe images")
    except:
        print("  Could not load extra safe images, using what we have")

print(f"  Total safe images: {len(safe_images)}")
save_split(safe_images, 'safe', prefix='safe')

# ============================================================
# Summary
# ============================================================
print("\n" + "=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
for split_name, split_dir in [('train', TRAIN_DIR), ('val', VAL_DIR)]:
    for cls in CLASSES:
        count = len(list((split_dir / cls).glob('*')))
        print(f'  {split_name}/{cls}: {count}')

# Warn about imbalances
train_counts = {cls: len(list((TRAIN_DIR / cls).glob('*'))) for cls in CLASSES}
min_count = min(train_counts.values())
max_count = max(train_counts.values())
if max_count > 5 * min_count and min_count > 0:
    print(f"\n⚠️ Class imbalance detected (ratio {max_count/min_count:.1f}x)")
    print("  WeightedRandomSampler in Cell 3 will handle this.")

In [ ]:
# Cell 3: Data augmentation & DataLoaders
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import numpy as np

# IMPORTANT: Use same normalization as sandbox.js in the extension!
# sandbox.js uses: (pixel / 255 - 0.5) / 0.5 → mean=0.5, std=0.5
# This MUST match or inference will be wrong.
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

train_dataset = datasets.ImageFolder('dataset/train', transform=train_transform)
val_dataset = datasets.ImageFolder('dataset/val', transform=val_transform)

print('Class to index mapping:', train_dataset.class_to_idx)
CLASSES = list(train_dataset.class_to_idx.keys())
NUM_CLASSES = len(CLASSES)
print(f'Detected {NUM_CLASSES} classes: {CLASSES}')

# Verify expected order (ImageFolder sorts alphabetically)
expected = {'gore': 0, 'horror': 1, 'nsfw': 2, 'safe': 3}
if train_dataset.class_to_idx != expected:
    print(f'⚠️ Class order differs from expected: {expected}')
    print(f'   Actual: {train_dataset.class_to_idx}')
    print('   sandbox.js CLASS_NAMES must match this order!')

# WeightedRandomSampler for class imbalance
targets = train_dataset.targets
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = class_weights[targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

print(f'\nClass counts: {dict(zip(CLASSES, class_counts))}')
print(f'Class weights: {dict(zip(CLASSES, class_weights.round(6)))}')

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nNormalization: mean={NORM_MEAN}, std={NORM_STD} (matches sandbox.js)')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

In [ ]:
# Cell 4: Model definition (EfficientNet-B0 fine-tune)
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Freeze early layers (features[:5]) to preserve low/mid-level features
# Layers 5-8 will be fine-tuned for our domain
for i, block in enumerate(model.features[:5]):
    for param in block.parameters():
        param.requires_grad = False

# Replace classifier: Dropout(0.3) -> Linear(1280, NUM_CLASSES)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(1280, NUM_CLASSES)
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Device: {device}')
print(f'Total params: {total_params:,}')
print(f'Trainable: {trainable_params:,} ({trainable_params/total_params:.1%})')
print(f'Output classes: {NUM_CLASSES} → {CLASSES}')

In [ ]:
# Cell 5: Training loop
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

backbone_params = [p for name, p in model.named_parameters() 
                   if p.requires_grad and 'classifier' not in name]
head_params = [p for name, p in model.named_parameters() 
               if 'classifier' in name]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params, 'lr': 1e-3},
], weight_decay=0.01)

MAX_EPOCHS = 30
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

class_weights_tensor = torch.FloatTensor(class_weights / class_weights.sum() * NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

PATIENCE = 5
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

print(f'Training for max {MAX_EPOCHS} epochs with early stopping (patience={PATIENCE})')
print(f'Backbone LR: 1e-5, Head LR: 1e-3')
print('-' * 60)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

for epoch in range(MAX_EPOCHS):
    # Training
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    scheduler.step()
    
    print(f'Epoch {epoch+1:2d}/{MAX_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}', end='')
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_model_state = model.state_dict().copy()
        torch.save(best_model_state, 'best_model.pth')
        print(' *best*')
    else:
        patience_counter += 1
        print(f' (patience {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

model.load_state_dict(best_model_state)
print(f'\nBest validation loss: {best_val_loss:.4f}')

In [ ]:
# Cell 6: Evaluation
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print('Classification Report:')
print('=' * 60)
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=3))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

accuracy = sum(1 for p, l in zip(all_preds, all_labels) if p == l) / len(all_labels)
print(f'\nOverall Validation Accuracy: {accuracy:.1%}')
if accuracy >= 0.90:
    print('Target met (>90%)')
else:
    print('Below target. Consider: more data, unfreezing more layers, or longer training.')

In [ ]:
# Cell 7: ONNX export
# CRITICAL: tensor names must match sandbox.js in the extension
# sandbox.js expects: input="pixel_values", output="logits"
import onnx

model.eval()
model_cpu = model.to('cpu')

dummy_input = torch.randn(1, 3, 224, 224)
ONNX_PATH = 'safeview_model.onnx'

torch.onnx.export(
    model_cpu,
    dummy_input,
    ONNX_PATH,
    opset_version=14,  # opset 14+ needed for some ops
    input_names=['pixel_values'],   # MUST match sandbox.js
    output_names=['logits'],        # MUST match sandbox.js
    dynamic_axes={
        'pixel_values': {0: 'batch_size'},
        'logits': {0: 'batch_size'}
    }
)

onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
print(f'ONNX model exported: {ONNX_PATH} ({size_mb:.1f} MB)')
print(f'Input name: "pixel_values", Output name: "logits"')
print(f'Class order: {CLASSES}')

# Quick inference test with ONNX Runtime
import onnxruntime as ort
session = ort.InferenceSession(ONNX_PATH)
result = session.run(None, {'pixel_values': dummy_input.numpy()})
print(f'\nTest output shape: {result[0].shape}')
print(f'Test logits: {result[0][0].round(3)}')

# Show softmax probabilities
import torch.nn.functional as F
probs = F.softmax(torch.tensor(result[0][0]), dim=0)
for cls, prob in zip(CLASSES, probs):
    print(f'  {cls}: {prob:.3f}')

In [ ]:
# Cell 8: Quantization (reduces ~20MB → ~5-7MB)
from onnxruntime.quantization import quantize_dynamic, QuantType

QUANTIZED_PATH = 'safeview_model_quantized.onnx'

quantize_dynamic(
    model_input=ONNX_PATH,
    model_output=QUANTIZED_PATH,
    weight_type=QuantType.QUInt8
)

orig_size = os.path.getsize(ONNX_PATH) / (1024 * 1024)
quant_size = os.path.getsize(QUANTIZED_PATH) / (1024 * 1024)
print(f'Original:  {orig_size:.1f} MB')
print(f'Quantized: {quant_size:.1f} MB ({quant_size/orig_size:.0%} of original)')

# Verify accuracy after quantization
print('\nVerifying quantized model accuracy...')
quant_session = ort.InferenceSession(QUANTIZED_PATH)

quant_preds = []
quant_labels = []

for images, labels in val_loader:
    result = quant_session.run(None, {'pixel_values': images.numpy()})
    preds = result[0].argmax(axis=1)
    quant_preds.extend(preds)
    quant_labels.extend(labels.numpy())

quant_accuracy = sum(1 for p, l in zip(quant_preds, quant_labels) if p == l) / len(quant_labels)
print(f'Quantized model accuracy: {quant_accuracy:.1%}')
print(f'Accuracy drop from FP32: {(accuracy - quant_accuracy):.2%}')

if quant_accuracy >= 0.88:
    print('✅ Quantized model quality is acceptable — use this one!')
    USE_MODEL = QUANTIZED_PATH
else:
    print('⚠️ Significant accuracy drop. Using FP32 model instead.')
    USE_MODEL = ONNX_PATH

print(f'\n→ Recommended model: {USE_MODEL} ({os.path.getsize(USE_MODEL) / (1024*1024):.1f} MB)')

In [ ]:
# Cell 9: Download the model
from google.colab import files

model_size = os.path.getsize(USE_MODEL) / (1024 * 1024)
print(f'Downloading: {USE_MODEL} ({model_size:.1f} MB)')
print()
print('After download, place the file in your extension:')
print('  1. Rename the downloaded file to: safeview_model.onnx')
print('  2. Copy to: safeview-extension/src/model/safeview_model.onnx')
print('  3. Run: npm run build')
print('  4. Reload extension in chrome://extensions')
print()
print(f'Class order in model: {CLASSES}')
print('This must match CLASS_NAMES in sandbox.js!')

files.download(USE_MODEL)